# Mission 2 eGeMAPSv02 88D — 공유 Training ZIP으로 Colab resume
모든 셀은 사용자가 위에서 아래로 직접 실행한다. 전체 extraction은 G에서만 실행한다.
팀 공유 폴더: https://drive.google.com/drive/folders/1cL-a0SsbT8Gxo1OSLc3eTy3Rccf2zeBo
확인된 파일은 `대학부 데이터-20260901T090511Z-1-001.zip`–`012.zip` 12개다. ZIP 내부 구조는 D에서 실제로 검사하며 추측으로 basename을 매칭하지 않는다.

공유 폴더가 mount에서 안 보이면 접근 가능한 계정의 My Drive에 **폴더 바로가기**를 추가하고 C의 경로를 지정한다. 원본 WAV/JSON source는 이 공유 ZIP 12개로 통일하며 기존에 풀어 둔 split 폴더는 원본으로 사용하지 않는다. manifest의 Training WAV/JSON만 ZIP에서 `/content`에 임시 staging해 기존 loader/crop/padding/해시 검사를 그대로 사용한다. 런타임이 초기화되면 임시 raw만 다시 준비하고 Drive의 working partial에서 이어간다.

**로컬 준비:** Windows extraction을 중지하고 기존 470255 completed partial의 일관된 snapshot, 기존 manifest/metadata, clean chain, `tmp/mission2_egemaps_transfer_identity.json`, 수정된 repo src/requirements만 Drive에 준비한다. 로컬 원본 WAV/JSON을 재업로드하지 않는다. transfer identity가 없다면 기존 `scripts/mission2_egemaps_transfer_identity.py`의 `--raw-root`에도 공유 ZIP에서 staging한 파일만 지정하고 사전에 추출된 split 폴더는 지정하지 않는다. 원래 partial 생성 시점의 byte 동일성은 소급 증명되지 않으므로 transfer identity와 overlap을 모두 유지한다.


## Checkpoint A — 환경 및 package 확인
누락된 package가 있으면 아래 별도 설치 셀을 사람이 실행하고 이 셀을 다시 실행한다. 자동 설치하지 않는다.


In [ ]:
import importlib.metadata as md
import os, platform, sys
required = ('numpy', 'pandas', 'scipy', 'scikit-learn', 'opensmile', 'pyarrow')
print('Python:', sys.version)
print('Platform:', platform.platform(), 'OS:', os.name, 'CPU count:', os.cpu_count())
missing = []
for package in required:
    try:
        print(package, md.version(package))
    except md.PackageNotFoundError:
        missing.append(package)
print('Missing:', missing)


### Checkpoint A 선택 설치 셀
위 셀에 누락 package가 있거나 버전 조정이 필요할 때만 직접 실행한다. 설치 후 런타임을 재시작하고 A부터 다시 실행한다. openSMILE 버전은 반드시 다시 확인한다.


In [ ]:
# 필요할 때만 직접 실행하고 런타임 재시작 후 A부터 다시 확인한다.
%pip install numpy pandas scipy scikit-learn opensmile==2.6.0 pyarrow


## B — Drive mount와 작업 경로
기존 `PARTIAL_ROOT` snapshot은 읽기 전용으로 보관한다. resume는 D에서 snapshot을 복사한 `WORKING_PARTIAL_ROOT`만 수정한다. `PROJECT_ROOT`, snapshot/identity 경로는 실제 Drive 위치를 사용한다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
PROJECT_ROOT = Path("/content/drive/MyDrive/prelim")
REPO_ROOT = PROJECT_ROOT
SHARED_ARCHIVE_ROOT = (
    PROJECT_ROOT
    / "data"
    / "raw_original"
    / "대학부 데이터"
)
RAW_ROOT = Path("/content/mission2_shared_training_raw")
TMP_ROOT = PROJECT_ROOT / "tmp"
PARTIAL_ROOT = TMP_ROOT / "mission2_egemaps_partial_recovery"
IDENTITY_PATH = TMP_ROOT / "mission2_egemaps_transfer_identity.json"
MANIFEST_PATH = TMP_ROOT / 'mission2_eda/utterance_manifest_v1.parquet'
MANIFEST_METADATA_PATH = TMP_ROOT / 'mission2_eda/utterance_manifest_meta_v1.json'
CHAIN_PATH = PROJECT_ROOT / 'data/processed/mission2_stage4_clean/stage4_clean_v1_090422f783f0d911/chain.json'
PARTIAL_SNAPSHOT_ROOT = PARTIAL_ROOT  # 470255행 원본, 수정 금지
WORKING_PARTIAL_ROOT = TMP_ROOT / 'mission2_egemaps_colab_working_partial'  # resume 작업 복사본
FINAL_PATH = PROJECT_ROOT / 'data/processed/mission2_egemaps_colab_v1' / 'training_egemaps_v02_88d.npz'
RECOVERY_FILES = ('matrix_88_partial.npy', 'completed_mask.npy', 'recovery_metadata.json', 'record_keys.npy', 'call_ids.npy')
required_paths = [REPO_ROOT / 'src/dcc_data/mission2_egemaps_colab.py', REPO_ROOT / 'requirements.txt', MANIFEST_PATH, MANIFEST_METADATA_PATH, CHAIN_PATH, IDENTITY_PATH]
required_paths += [PARTIAL_SNAPSHOT_ROOT / name for name in RECOVERY_FILES]
missing_paths = [str(path) for path in required_paths if not path.is_file()]
if missing_paths: raise RuntimeError(f'STOP: missing Drive files: {missing_paths}')
if WORKING_PARTIAL_ROOT.resolve() == PARTIAL_SNAPSHOT_ROOT.resolve(): raise RuntimeError('STOP: working partial must differ from snapshot')
if FINAL_PATH.exists(): raise FileExistsError(f'STOP: final cache exists: {FINAL_PATH}')
print('Drive paths PASS:', len(required_paths), 'final path:', FINAL_PATH)


## C — shared data source 설정
`SHARED_ARCHIVE_ROOT`에서 ZIP 12개를 확인하고 원본 WAV/JSON은 이 archive에서만 시작한다. 다음 source 준비 셀에서 manifest의 Training 항목만 `RAW_ROOT`에 staging한다. 그 뒤 모든 WAV reader는 `RAW_ROOT`만 사용한다. shell unzip이나 extractall은 사용하지 않는다.


In [ ]:
ARCHIVE_PATHS = [SHARED_ARCHIVE_ROOT / f'대학부 데이터-20260901T090511Z-1-{number:03d}.zip' for number in range(1, 13)]
missing_archives = [str(path) for path in ARCHIVE_PATHS if not path.is_file()]
if missing_archives: raise RuntimeError(f'STOP: shared ZIP files missing; check shortcut/account/path: {missing_archives}')
if not RAW_ROOT.resolve().is_relative_to(Path('/content')) or RAW_ROOT.resolve().is_relative_to(Path('/content/drive')):
    raise RuntimeError('STOP: RAW_ROOT must be temporary local /content storage outside Drive')
for archive in ARCHIVE_PATHS:
    print(archive.name, 'bytes:', archive.stat().st_size)


## D — snapshot과 working partial strict validation
889358×88, completed 470255, remaining 419103, key/call/feature/padding/fingerprint를 snapshot에서 읽기 전용으로 확인한다. 새 working 폴더가 없을 때만 snapshot의 다섯 파일을 복사한다. 기존 working 폴더가 불완전하면 STOP하며 자동 재시작하지 않는다.


In [ ]:
import json, sys
sys.path.insert(0, str(REPO_ROOT / 'src'))
import numpy as np, pandas as pd
from dcc_data.mission2_egemaps_colab import EXPECTED_FINGERPRINT, environment_provenance, validate_existing_partial, get_egemaps_feature_names
environment = environment_provenance()
print('Runtime provenance:', environment)
if environment['packages']['opensmile'] != '2.6.0': raise RuntimeError('STOP: local openSMILE contract requires 2.6.0')
with CHAIN_PATH.open(encoding='utf-8') as stream: chain = json.load(stream)
if chain.get('provenance_fingerprint') != EXPECTED_FINGERPRINT: raise RuntimeError('STOP: clean chain fingerprint differs')
with MANIFEST_METADATA_PATH.open(encoding='utf-8') as stream: manifest_metadata = json.load(stream)
if manifest_metadata.get('schema_version') != 'mission2_raw_eda_v1': raise RuntimeError('STOP: source manifest schema differs')
manifest_all = pd.read_parquet(MANIFEST_PATH)
training = manifest_all.loc[manifest_all['split'].astype(str) == 'Training'].copy()
keys = np.load(PARTIAL_SNAPSHOT_ROOT / 'record_keys.npy', allow_pickle=True).astype(str)
calls = np.load(PARTIAL_SNAPSHOT_ROOT / 'call_ids.npy', allow_pickle=True).astype(str)
if len(keys) != 889358 or len(set(keys)) != len(keys): raise RuntimeError('STOP: key count/uniqueness differs')
manifest = training.reset_index(drop=True)  # clean chain uses the original Training order
if not np.array_equal(manifest['record_key'].to_numpy(), keys): raise RuntimeError('STOP: partial differs from clean Training record_key order')
names = get_egemaps_feature_names()
snapshot = validate_existing_partial(manifest, PARTIAL_SNAPSHOT_ROOT, keys, calls, names, fingerprint=EXPECTED_FINGERPRINT)
if snapshot['completed'] != 470255 or snapshot['remaining'] != 419103:
    raise RuntimeError(f'STOP: original Windows partial count differs: {snapshot}')
import shutil
if not WORKING_PARTIAL_ROOT.exists():
    WORKING_PARTIAL_ROOT.mkdir(parents=True, exist_ok=False)
    for filename in RECOVERY_FILES:
        shutil.copyfile(PARTIAL_SNAPSHOT_ROOT / filename, WORKING_PARTIAL_ROOT / filename)
print('SNAPSHOT_VALIDATION_PASS', snapshot)


### D — working partial 중단 복구와 원본 행 보존 검사
기존 atomic journal이 있을 때 working partial의 미확정 행만 복구한다. snapshot은 수정하지 않는다. snapshot에서 완료된 470255개 행이 working partial에서도 완료 상태이고 값이 동일해야 한다.


In [ ]:
from dcc_data.mission2_egemaps_colab import recover_interrupted_checkpoint
if any(not (WORKING_PARTIAL_ROOT / filename).is_file() for filename in RECOVERY_FILES):
    raise RuntimeError('STOP: working partial copy incomplete; preserve it and inspect before continuing')
print(recover_interrupted_checkpoint(WORKING_PARTIAL_ROOT))
partial = validate_existing_partial(manifest, WORKING_PARTIAL_ROOT, keys, calls, names, fingerprint=EXPECTED_FINGERPRINT)
snapshot_mask = np.load(PARTIAL_SNAPSHOT_ROOT / 'completed_mask.npy', mmap_mode='r')
working_mask = np.load(WORKING_PARTIAL_ROOT / 'completed_mask.npy', mmap_mode='r')
snapshot_matrix = np.load(PARTIAL_SNAPSHOT_ROOT / 'matrix_88_partial.npy', mmap_mode='r')
working_matrix = np.load(WORKING_PARTIAL_ROOT / 'matrix_88_partial.npy', mmap_mode='r')
for start in range(0, len(keys), 8192):
    stop = min(start + 8192, len(keys))
    completed = snapshot_mask[start:stop]
    if not working_mask[start:stop][completed].all() or not np.array_equal(snapshot_matrix[start:stop][completed], working_matrix[start:stop][completed]):
        raise RuntimeError('STOP: original Windows completed rows changed in working partial')
del snapshot_mask, working_mask, snapshot_matrix, working_matrix
print('PARTIAL_VALIDATION_PASS', partial, 'original 470255 completed rows preserved')


### D — ZIP 내부 경로 검사, Training만 임시 해제, source identity, 실제 WAV 1개 검증
먼저 12개 ZIP의 실제 member 예시를 출력한다. 그다음 manifest 기준 29200 WAV + 29200 JSON의 누락·중복·경로·call alignment를 검사한다. 실제 필요한 해제 용량 + 2 GiB 여유가 없으면 STOP한다. 기존 임시 파일도 내용이 다르면 덮어쓰지 않는다.
공유 ZIP 내부를 아직 확인하지 못한 상태에서 작성된 안전한 검사다. 내부에 다시 ZIP이 있거나 한글 encoding/경로가 다르면 STOP하며 결과를 확인해야 한다. 준비 후 로컬 transfer identity의 58400개 원본 SHA-256이 모두 일치해야 overlap/resume로 진행한다. 이 셀은 partial에 추출값을 쓰지 않는다.


In [ ]:
import zipfile
from dcc_data.mission2_egemaps_colab import inspect_shared_training_archives, prepare_shared_training_raw, verify_transfer_identity
for archive_path in ARCHIVE_PATHS:
    with zipfile.ZipFile(archive_path) as archive:
        members = [item.filename for item in archive.infolist() if not item.is_dir()]
    print(archive_path.name, 'member count:', len(members), 'examples:', members[:3])
source_plan = inspect_shared_training_archives(manifest, ARCHIVE_PATHS)
if source_plan['wav_files'] != 29200 or source_plan['json_files'] != 29200 or source_plan['files'] != 58400:
    raise RuntimeError('STOP: Training raw coverage differs from original manifest')
print('SOURCE_INDEX_PASS', {key: value for key, value in source_plan.items() if key != 'members'})
print('Training uncompressed GiB:', source_plan['uncompressed_bytes'] / 1024**3)
staged = prepare_shared_training_raw(manifest, ARCHIVE_PATHS, RAW_ROOT)
print('TRAINING_STAGE_PASS', staged)
identity = verify_transfer_identity(manifest, MANIFEST_PATH, MANIFEST_METADATA_PATH, RAW_ROOT, IDENTITY_PATH)
print('SOURCE_IDENTITY_PASS', identity)
from dcc_data.audio import read_wav_frames
from dcc_data.mission2_egemaps_resume import extract_with_short_audio_fallback
row = manifest.iloc[0]
audio_path = RAW_ROOT / row['audio_path']
audio = read_wav_frames(audio_path)
if audio.sample_rate != 8000 or audio.waveform.shape[0] != 1:
    raise RuntimeError('STOP: shared Training WAV must be native 8 kHz mono')
start, end = int(row['crop_start_frame']), int(row['crop_end_frame'])
if start < 0 or end <= start or end > audio.waveform.shape[1]: raise RuntimeError('STOP: manifest crop invalid')
vector, returned_names, padded = extract_with_short_audio_fallback(audio.waveform[0, start:end], audio.sample_rate)
if vector.shape != (88,) or vector.dtype != np.float32 or not np.isfinite(vector).all() or returned_names != names:
    raise RuntimeError('STOP: shared WAV eGeMAPSv02 88D contract differs')
print('SHARED_WAV_SMOKE_PASS', row['audio_path'], 'call_id:', row['call_id'], 'record_key:', row['record_key'], 'Hz:', audio.sample_rate, 'shape:', vector.shape, 'padded:', padded)
checked = validate_existing_partial(manifest, WORKING_PARTIAL_ROOT, keys, calls, names, fingerprint=EXPECTED_FINGERPRINT)
if checked != partial: raise RuntimeError('STOP: partial changed during source/smoke validation')
del source_plan, vector, audio


## E — Windows/Colab 100행 overlap audit
완료된 행을 길이별로 결정적으로 100개 고르고, 완료된 padded 행도 포함한다. Windows 저장값과 Colab 재추출값을 exact 및 rtol=1e-7, atol=1e-8로 각각 비교한다. 이 tolerance는 자동 승인 기준이 아니다.


In [ ]:
from dcc_data.mission2_egemaps_colab import audit_overlap
overlap = audit_overlap(manifest, RAW_ROOT, WORKING_PARTIAL_ROOT, keys, calls, names)
print('CROSS_PLATFORM_STATUS:', overlap['status'])
for field in ('sampled_rows', 'sampled_calls', 'padded_sampled', 'exact_row_count', 'numerically_close_row_count', 'differing_row_count', 'differing_value_count', 'mean_abs_diff', 'max_abs_diff', 'top_changed_features'):
    print(field, overlap[field])


## F — 선택: 실제 250발화 benchmark
속도를 확인할 때만 RUN_BENCHMARK를 True로 바꾼다. 미완료 발화를 읽고 추출하지만 partial/cache에는 쓰지 않는다. ETA는 단순 추정이다.


In [ ]:
from dcc_data.mission2_egemaps_colab import benchmark_extraction
RUN_BENCHMARK = False
benchmark = None
if RUN_BENCHMARK:
    benchmark = benchmark_extraction(manifest, RAW_ROOT, WORKING_PARTIAL_ROOT, keys, calls, names, count=250)
    print('BENCHMARK_ONLY', benchmark)
else:
    print('BENCHMARK_SKIPPED')


### F — overlap과 source identity에 따른 resume 승인 guard
D의 source identity PASS가 필요하다. E의 100행 exact면 통과한다. numerically close면 `APPROVE_NUMERICALLY_CLOSE`를 직접 입력한다. mismatch는 제한된 수치·source·partial 조건을 모두 만족하고 사용자가 `APPROVE_CROSS_PLATFORM_MISMATCH`를 직접 입력한 경우에만 resume한다. mismatch 판정은 유지된다.


In [ ]:
import importlib, dcc_data.mission2_egemaps_colab as egemaps_colab
importlib.reload(egemaps_colab)  # Drive에 반영된 최신 승인 guard를 사용한다.
from dcc_data.mission2_egemaps_colab import APPROVAL_PHRASE, MISMATCH_APPROVAL_PHRASE, approve_resume
with (WORKING_PARTIAL_ROOT / 'recovery_metadata.json').open(encoding='utf-8') as stream:
    approval_partial_metadata = json.load(stream)
approval_partial = {**partial, 'feature_dim': approval_partial_metadata['dim'], 'feature_names': approval_partial_metadata['feature_names'], 'resume_history': approval_partial_metadata.get('resume_history', [])}
explicit = None
if overlap['status'] == 'CROSS_PLATFORM_NUMERICALLY_CLOSE':
    print('Review the differences printed in E before approving.')
    explicit = input(f'Type {APPROVAL_PHRASE} to approve mixing: ')
elif overlap['status'] == 'CROSS_PLATFORM_FEATURE_MISMATCH':
    print('Manual mismatch override requires every guard to pass.')
    print('Overlap:', {key: overlap.get(key) for key in ('status', 'sampled_rows', 'sampled_calls', 'padded_sampled', 'exact_row_count', 'numerically_close_row_count', 'differing_row_count', 'differing_value_count', 'mean_abs_diff', 'max_abs_diff')})
    print('Source identity:', {key: identity.get(key) for key in ('status', 'source_split', 'raw_file_count', 'rows', 'clean_reference_fingerprint', 'raw_content_sha256')})
    explicit = input(f'Type {MISMATCH_APPROVAL_PHRASE} to approve this constrained resume: ')
approval = approve_resume(approval_partial, overlap, identity, explicit_approval=explicit, environment=environment)
print('RESUME_APPROVED', approval, 'benchmark:', benchmark)


## G — remaining rows만 resume extraction
F 승인 객체가 필요하다. 470255개 Windows 완료 행을 재추출하거나 덮어쓰지 않고 working partial의 미완료 행만 채운다. 250 calls마다 working partial을 checkpoint한다. 런타임 종료 후 A–F를 다시 실행한다. /content raw가 사라졌으면 D에서 다시 준비하고 Drive working partial을 이어쓴다. snapshot은 보관한다.


In [ ]:
from dcc_data.mission2_egemaps_colab import resume_approved_partial
if approval.get('approval_type') == 'APPROVE_CROSS_PLATFORM_MISMATCH':
    print('START_COMPLETED =', approval['starting_completed_rows'])
    print('START_REMAINING =', approval['starting_remaining_rows'])
    snapshot_mask = np.load(PARTIAL_SNAPSHOT_ROOT / 'completed_mask.npy', mmap_mode='r')
    working_mask = np.load(WORKING_PARTIAL_ROOT / 'completed_mask.npy', mmap_mode='r')
    snapshot_matrix = np.load(PARTIAL_SNAPSHOT_ROOT / 'matrix_88_partial.npy', mmap_mode='r')
    working_matrix = np.load(WORKING_PARTIAL_ROOT / 'matrix_88_partial.npy', mmap_mode='r')
    for start in range(0, len(keys), 8192):
        stop = min(start + 8192, len(keys))
        completed = snapshot_mask[start:stop]
        if not working_mask[start:stop][completed].all() or not np.array_equal(snapshot_matrix[start:stop][completed], working_matrix[start:stop][completed]):
            raise RuntimeError('STOP: original Windows completed rows changed before resume')
    del snapshot_mask, working_mask, snapshot_matrix, working_matrix
progress = resume_approved_partial(manifest, RAW_ROOT, WORKING_PARTIAL_ROOT, keys, calls, names, approval, overlap, identity, environment)
print('RESUME_CHECKPOINT', progress)


## H — final strict validation과 별도 경로 publish
889358행 모두 완료·finite·float32·88D이고 key/call/feature/padding provenance가 같을 때만 새 final cache를 저장한다. mismatch 수동승인 provenance와 G history를 확인하고 Windows snapshot의 기존 470255 completed rows도 finalization 직전에 다시 비교한다. partial과 Windows cache는 남긴다.


In [ ]:
from dcc_data.mission2_egemaps_colab import publish_approved_cache
published = publish_approved_cache(manifest, WORKING_PARTIAL_ROOT, FINAL_PATH, keys, calls, names, approval, overlap, identity, environment, snapshot_root=PARTIAL_SNAPSHOT_ROOT)
print('FINAL_CACHE_PUBLISHED', published)
